# 02 - Modeling

Train models for credit card fraud detection and evaluate them with precision-recall metrics.

## Setup

Add the project root to the path so we can import from `src`.

In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

from sklearn.metrics import average_precision_score

from src.data import load_processed, split_data
from src.evaluate import metrics_at
from src.features import add_features
from src.train import train_baseline

## Data

Load the processed data and make the stratified train/test split. The engineered features are added after the split, separately on each set.

In [2]:
X_train, X_test, y_train, y_test = split_data(load_processed(), method="random")
X_train = add_features(X_train)
X_test = add_features(X_test)

print("Train:", X_train.shape, "Test:", X_test.shape)

Train: (226980, 30) Test: (56746, 30)


## Baseline: Logistic Regression

`StandardScaler` + `LogisticRegression(class_weight="balanced")`. This is a simple reference point to compare against the Random Forest later.

We report PR-AUC (average precision) and precision, recall and F1 at the default threshold of 0.5. The threshold is fixed, not tuned on the test set.

In [3]:
baseline = train_baseline(X_train, y_train)
baseline_proba = baseline.predict_proba(X_test)[:, 1]

baseline_metrics = metrics_at(y_test, baseline_proba, threshold=0.5)
baseline_metrics["pr_auc"] = average_precision_score(y_test, baseline_proba)

baseline_metrics

{'threshold': 0.5,
 'precision': 0.05471324983520105,
 'recall': 0.8736842105263158,
 'f1': 0.10297766749379653,
 'tp': 83,
 'fp': 1434,
 'fn': 12,
 'tn': 55217,
 'pr_auc': 0.6809630745926131}

## Random Forest: comparing `class_weight`

We compare three `class_weight` settings with stratified 3-fold cross-validation **on the training set only** (the test set is not touched). Each model uses 100 trees to keep it fast. The score is PR-AUC (average precision).

In [4]:
import pandas as pd
from sklearn.model_selection import StratifiedKFold, cross_val_score

from src.config import SEED
from src.train import make_rf

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)

rows = []
for class_weight in [None, "balanced", "balanced_subsample"]:
    model = make_rf(n_estimators=100, class_weight=class_weight)
    scores = cross_val_score(model, X_train, y_train, cv=cv, scoring="average_precision")
    rows.append(
        {
            "class_weight": str(class_weight),
            "pr_auc_mean": scores.mean(),
            "pr_auc_std": scores.std(),
        }
    )

cv_results = pd.DataFrame(rows).round(4)
cv_results

,class_weight,pr_auc_mean,pr_auc_std
0,None,0.8334,0.0177
1,balanced,0.8286,0.0161
2,balanced_subsample,0.8260,0.0117


### Result

The numbers in the next cell are generated from `cv_results` and `baseline_metrics`, so they always match the outputs above.

In [5]:
from IPython.display import Markdown, display

by_weight = cv_results.set_index("class_weight")
none, subsample = by_weight.loc["None"], by_weight.loc["balanced_subsample"]
diff = none["pr_auc_mean"] - subsample["pr_auc_mean"]
smaller_std = min(none["pr_auc_std"], subsample["pr_auc_std"])

if abs(diff) < smaller_std:
    verdict = "smaller than the fold-to-fold standard deviation of either setting, so it is within the noise"
else:
    verdict = (
        "at least as large as the fold-to-fold standard deviation, so it may be a real difference"
    )

if diff >= 0:
    comparison = "was not worse than"
else:
    comparison = "scored lower than"

rf_low, rf_high = cv_results["pr_auc_mean"].min(), cv_results["pr_auc_mean"].max()
baseline_pr_auc = baseline_metrics["pr_auc"]
rf_vs_baseline = "above" if rf_low > baseline_pr_auc else "not clearly above"

display(
    Markdown(f"""
- Mean PR-AUC across the three settings ranges from **{rf_low:.4f}** to **{rf_high:.4f}**,
  with fold standard deviations from {cv_results["pr_auc_std"].min():.4f} to
  {cv_results["pr_auc_std"].max():.4f}.
- `class_weight=None` ({none["pr_auc_mean"]:.4f} ± {none["pr_auc_std"]:.4f}) {comparison}
  `balanced_subsample` ({subsample["pr_auc_mean"]:.4f} ± {subsample["pr_auc_std"]:.4f}).
  The difference ({diff:+.4f}) is {verdict}.
- All Random Forest settings ({rf_low:.4f}-{rf_high:.4f}) are {rf_vs_baseline} the Logistic
  Regression baseline (PR-AUC {baseline_pr_auc:.4f}).
""")
)


- Mean PR-AUC across the three settings ranges from **0.8260** to **0.8334**,
  with fold standard deviations from 0.0117 to
  0.0177.
- `class_weight=None` (0.8334 ± 0.0177) was not worse than
  `balanced_subsample` (0.8260 ± 0.0117).
  The difference (+0.0074) is smaller than the fold-to-fold standard deviation of either setting, so it is within the noise.
- All Random Forest settings (0.8260-0.8334) are above the Logistic
  Regression baseline (PR-AUC 0.6810).


With 3 folds and 100 trees, **this comparison does not show that any `class_weight` option is better**. With only 3 folds the standard deviation itself is a rough estimate, so small differences should not be over-read.

`balanced_subsample` is kept in the final model because it is the conventional choice for a Random Forest on heavily imbalanced data, **not because it won this comparison**: `None` did not do worse here.

This is plausible: PR-AUC depends on the ranking of the predicted probabilities, and class weights mainly shift the probabilities rather than changing that ranking much. The choice of `class_weight` can still matter for the decision threshold, which is set separately.

The baseline comparison is only rough: the baseline PR-AUC was measured on the test set, while the Random Forest scores are cross-validation scores on the training set.